# SymPyによる組合せ論：離散数学の記号的計算

## 概要
組合せ論は、離散的な対象の数え上げや配置を扱う数学の一分野である。順列・組合せの計算から、母関数（Generating Function）を用いた漸化式の解法、整数分割の問題まで、その応用範囲は広い。

これらの問題は、手計算では複雑な式変形や級数展開を伴うことが多いが、Pythonの数式処理ライブラリ **SymPy** を用いれば、記号的に厳密な計算を行うことができる。特に、母関数を用いた解法では、無限級数の操作が必要となるため、SymPyの威力が発揮される。

本記事では、SymPyを用いて以下のトピックを解説する。

1. **基本的な組合せ計算**：階乗、順列、組合せの記号的表現
2. **母関数**：フィボナッチ数列の閉じた形の導出
3. **整数分割**：分割数の母関数
4. **漸化式の解法**：カタラン数の導出

### 筆者の環境
筆者の実行環境は以下の通りである。

In [ ]:
!sw_vers

In [ ]:
!python -V

必要なライブラリをインポートする。組合せ論では `factorial`, `binomial`, `rsolve` などが中心的な役割を果たす。

In [ ]:
import sympy
from sympy import symbols, factorial, binomial, Function, rsolve, simplify, expand, series, Sum, Product, oo
from sympy.abc import n, k, x
from pprint import pprint as py_pprint

# 数式をLaTeX形式で綺麗に表示するための設定
sympy.init_printing()

print("sympy version :", sympy.__version__)

## 1. 基本的な組合せ計算

組合せ論の基礎となる階乗 $n!$、順列 $_nP_r$、組合せ $_nC_r$ をSymPyで表現する。

### 1.1 階乗と二項係数

二項係数は以下のように定義される。

$$ \binom{n}{k} = \frac{n!}{k!(n-k)!} $$

In [ ]:
# 記号的な二項係数
binom_expr = binomial(n, k)

print("二項係数 C(n, k):")
sympy.pprint(binom_expr)

# 具体的な値
print("\nC(5, 2) =", binomial(5, 2))

### 1.2 二項定理の検証

二項定理は以下のように表される。

$$ (x+y)^n = \sum_{k=0}^n \binom{n}{k} x^{n-k} y^k $$

SymPyで展開して確認してみる。

In [ ]:
y = symbols('y')

# (x+y)^5 を展開
expanded = expand((x + y)**5)

print("(x+y)^5 の展開:")
sympy.pprint(expanded)

係数が二項係数と一致していることが確認できる。

## 2. 母関数によるフィボナッチ数列の解法

フィボナッチ数列は以下の漸化式で定義される。

$$ F_0 = 0, \quad F_1 = 1, \quad F_n = F_{n-1} + F_{n-2} \quad (n \geq 2) $$

母関数 $G(x) = \sum_{n=0}^\infty F_n x^n$ を用いて、閉じた形を導出する。

### 2.1 母関数の方程式

漸化式の両辺に $x^n$ をかけて $n$ について和をとると、

$$ G(x) - F_0 - F_1 x = x G(x) + x^2 G(x) $$

初期条件 $F_0=0, F_1=1$ を代入して整理すると、

$$ G(x) = \frac{x}{1 - x - x^2} $$

In [ ]:
# 母関数
G = x / (1 - x - x**2)

print("フィボナッチ数列の母関数:")
sympy.pprint(G)

### 2.2 部分分数分解と閉じた形

この有理関数を部分分数分解すると、ビネの公式が得られる。
まず分母を因数分解する。$1 - x - x^2 = 0$ の解は、

$$ x = \frac{-1 \pm \sqrt{5}}{2} $$

黄金比 $\phi = \frac{1+\sqrt{5}}{2}$ を用いると、

$$ G(x) = \frac{1}{\sqrt{5}} \left( \frac{1}{1 - \phi x} - \frac{1}{1 - (1-\phi) x} \right) $$

これを級数展開すると、

$$ F_n = \frac{\phi^n - (1-\phi)^n}{\sqrt{5}} $$

In [ ]:
from sympy import apart, sqrt

# 部分分数分解
G_partial = apart(G, x)

print("部分分数分解:")
sympy.pprint(G_partial)

## 3. 整数分割の母関数

整数 $n$ の分割数 $p(n)$ は、$n$ を正の整数の和として表す方法の総数である。
例えば $p(4) = 5$ である（$4, 3+1, 2+2, 2+1+1, 1+1+1+1$）。

分割数の母関数は以下のように表される。

$$ \sum_{n=0}^\infty p(n) x^n = \prod_{k=1}^\infty \frac{1}{1-x^k} $$

SymPyで有限項まで展開してみる。

In [ ]:
# 有限積で近似（k=1 to 10）
partition_gf = 1
for i in range(1, 11):
    partition_gf *= 1 / (1 - x**i)

# x^10 まで展開
partition_series = series(partition_gf, x, 0, 11)

print("分割数の母関数（x^10まで）:")
sympy.pprint(partition_series)

係数が分割数に対応している。例えば $x^4$ の係数は 5 となっているはずである。

## 4. 漸化式の解法：カタラン数

カタラン数 $C_n$ は、様々な組合せ問題（括弧の対応、二分木の数など）に現れる数列である。
以下の漸化式で定義される。

$$ C_0 = 1, \quad C_n = \sum_{k=0}^{n-1} C_k C_{n-1-k} $$

あるいは、閉じた形として、

$$ C_n = \frac{1}{n+1} \binom{2n}{n} $$

SymPyの `rsolve` を用いて、別の漸化式から導出してみる。

$$ C_n = \frac{4n-2}{n+1} C_{n-1}, \quad C_0 = 1 $$

In [ ]:
C = Function('C')

# 漸化式の定義
recurrence = C(n) - (4*n - 2) / (n + 1) * C(n - 1)

# 初期条件
init_cond = {C(0): 1}

# 解く
catalan_sol = rsolve(recurrence, C(n), init_cond)

print("カタラン数の一般項:")
sympy.pprint(simplify(catalan_sol))

結果は $\frac{1}{n+1} \binom{2n}{n}$ と一致するはずである。

具体的な値を計算してみる。

In [ ]:
# C_0 から C_5 まで計算
catalan_values = [catalan_sol.subs(n, i) for i in range(6)]

print("カタラン数（C_0 to C_5）:")
py_pprint(catalan_values)

結果は $[1, 1, 2, 5, 14, 42]$ となり、カタラン数列として知られる値と一致する。

## 結論

この記事では、SymPyを用いて組合せ論の基本的な問題から高度な母関数の解法までを扱った。

階乗や二項係数といった基本的な計算から、フィボナッチ数列の閉じた形の導出、整数分割の母関数、そしてカタラン数の漸化式の解法まで、離散数学における記号計算の威力を示した。特に、母関数を用いた解法では、無限級数の操作が必要となるため、手計算では困難な問題もSymPyを用いれば容易に解くことができる。

### 参考文献
- [SymPy Documentation: Combinatorics](https://docs.sympy.org/latest/modules/combinatorics/index.html)
- [SymPy Documentation: Concrete Mathematics](https://docs.sympy.org/latest/modules/concrete.html)